# Exploration des donnees — Dans Ma Rue (signalements citoyens)

Dans Ma Rue est l'application de signalement d'anomalies sur l'espace public de la Ville
de Paris (proprete, mobilier urbain degrade, occupation abusive de l'espace public...).
Ce notebook explore le jeu de donnees utilise pour l'analyse.

**Note sur le choix de la source** : Open Data Paris propose un jeu intitule
`dans-ma-rue-historique-anomalies-signalees`, qui semble a priori etre la source
naturelle. Il s'avere en realite qu'il ne s'agit que d'un conteneur de pieces jointes
(un fichier ZIP par annee depuis 2012), sans API de requete interrogeable
(`has_records: false` dans les metadonnees). Le jeu reellement exploitable, disponible
en direct et mis a jour quotidiennement, est `dans-ma-rue` (1,47 million de lignes) —
c'est celui utilise ici.

In [1]:
signalements = pd.read_csv("../data/dans-ma-rue.csv", sep=";")
signalements.shape

(1474285, 17)

In [1]:
signalements.info()

<class 'pandas.DataFrame'>
RangeIndex: 1474285 entries, 0 to 1474284
Data columns (total 17 columns):
 #   Column            Non-Null Count    Dtype
---  ------            --------------    -----
 0   numero            1474285 non-null  int64
 1   type              1474285 non-null  str
 2   soustype          1474281 non-null  str
 3   adresse           1474285 non-null  str
 4   code_postal       1474282 non-null  float64
 5   ville             1474285 non-null  str
 6   arrondissement    1474285 non-null  int64
 7   conseilquartier   1474174 non-null  str
 8   datedecl          1474285 non-null  str
 9   anneedecl         1474285 non-null  int64
 10  moisdecl          1474285 non-null  int64
 11  prefixe           1474285 non-null  str
 12  intervenant       1474283 non-null  str
 13  id_dmr            1474285 non-null  str
 14  geo_shape         1474285 non-null  str
 15  geo_point_2d      1474285 non-null  str
 16  mois_annee_decla  1474285 non-null  str
dtypes: float64(1), int64(4

Colonnes principales : `type`/`soustype` (nature du signalement), `adresse` (deja
geocodee, avec code postal et ville), `arrondissement` (entier, contrairement au format
"code postal" de `terrasses`), `conseilquartier`, `datedecl`/`moisdecl`/`anneedecl`
(date du signalement), et les coordonnees geographiques.

In [1]:
signalements['type'].value_counts()

type
Objets abandonnes                             631217
Graffitis, tags, affiches et autocollants      342448
Voirie et espace public                         55510
Proprete                                       243646
Activites commerciales et professionnelles      30912
Mobiliers urbains                               44140
Autos, motos, velos, trottinettes...            86502
Arbres, vegetaux et animaux                     18657
Eau                                              7996
Eclairage / Electricite                         13256
Degradation du sol                                  1
Name: count, dtype: int64

In [1]:
signalements['adresse'].head()

0                16 rue Victor Duruy, 75015 PARIS
1         31 Boulevard Saint-Jacques, 75014 PARIS
2                      2 Rue d'Orsel, 75018 PARIS
3                63 Quai de la Seine, 75019 PARIS
4      8 Boulevard de Bonne Nouvelle, 75010 PARIS
Name: adresse, dtype: str

Les adresses sont ici **geocodees automatiquement** : casse mixte, accents, et un
suffixe `", 75XXX PARIS"` que `terrasses` n'a pas. Deux formats a harmoniser avant
toute jointure (voir le notebook d'analyse principal).

In [1]:
signalements['arrondissement'].value_counts().sort_index()

arrondissement
0        102
1      23680
2      24986
3      52893
4      46020
5      32249
6      21623
7      22705
8      37317
9      61152
10    105243
11    112121
12     88825
13     80375
14     64846
15    121025
16     76525
17    125147
18    132190
19    113396
20    131858
44         3
82         4
Name: count, dtype: int64

Trois codes invalides apparaissent (`0`, `44`, `82`) — Paris ne compte que 20
arrondissements (1 a 20). Ils concernent 109 lignes sur 1,47 million (0,007 %),
negligeables en volume mais a exclure explicitement de toute agregation par
arrondissement (voir notebook d'analyse principal).